In [2]:
import pandas as pd

# Load cleaned data if available (created in 1.09), else fallback to raw train.csv
data_frame = pd.read_csv('output/1.09_cleaned_train.csv')


In [3]:
import numpy as np

df = data_frame.copy()
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Age__was_missing,Cabin__was_missing,Embarked__was_missing
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1.0,0,A/5 21171,7.2500,B96 B98,S,0,1,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1.0,0,PC 17599,65.6344,C85,C,0,0,0
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0.0,0,STON/O2. 3101282,7.9250,B96 B98,S,0,1,0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1.0,0,113803,53.1000,C123,S,0,0,0
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0.0,0,373450,8.0500,B96 B98,S,0,1,0


# 1.10 Data Preprocessing (Course Demonstration)

We demonstrate these techniques:
- **1.1 Data Selection**
- **1.2 Feature Selection**
- **1.3 Deriving New Features**
- **1.4 One-Hot Encoding / Dummy Coding**
- **1.5 Ordinal / Label Encoding**
- **1.6 Data Transformation (Scaling / Log)**

## 1.1 Data Selection

Selecting relevant data usually means:
- Removing columns with many missing values
- Removing constant columns
- Filtering rows with impossible values

In [4]:
df.shape, df.columns.tolist()[:20]

((891, 15),
 ['PassengerId',
  'Survived',
  'Pclass',
  'Name',
  'Sex',
  'Age',
  'SibSp',
  'Parch',
  'Ticket',
  'Fare',
  'Cabin',
  'Embarked',
  'Age__was_missing',
  'Cabin__was_missing',
  'Embarked__was_missing'])

In [5]:
# Drop columns with too many missing values (keep columns with >= 70% non-missing)
col_threshold = int(0.7 * len(df))
df_sel = df.dropna(axis=1, thresh=col_threshold).copy()

# Drop constant columns
constant_cols = [c for c in df_sel.columns if df_sel[c].nunique(dropna=False) <= 1]
df_sel = df_sel.drop(columns=constant_cols)

df_sel.shape

(891, 15)

In [6]:
# Example row filtering (only runs if a common column exists)
df_rows = df_sel.copy()
if 'Age' in df_rows.columns:
    df_rows = df_rows[(df_rows['Age'] >= 0) & (df_rows['Age'] <= 100)].copy()
df_rows.shape

(891, 15)

## 1.2 Feature Engineering (Feature Selection)

Simple demo: remove **highly correlated** numeric features (redundant info).

In [7]:
df_fs = df_rows.copy()
num_cols = df_fs.select_dtypes(include=[np.number]).columns.tolist()

if len(num_cols) >= 2:
    corr = df_fs[num_cols].corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [c for c in upper.columns if any(upper[c] > 0.90)]
    df_fs = df_fs.drop(columns=to_drop)

df_fs.shape

(891, 15)

In [8]:
# If a common numeric target exists, show the most correlated numeric features
target_candidates = ['Survived', 'target', 'Target', 'label', 'y']
target = next((c for c in target_candidates if c in df_fs.columns), None)

if target and target in df_fs.select_dtypes(include=[np.number]).columns:
    corr_with_target = df_fs.select_dtypes(include=[np.number]).corr()[target].sort_values(ascending=False)
    corr_with_target.head(10)
else:
    print('No numeric target column found (this is OK).')

## 1.3 Feature Engineering (Deriving New Data)

Create new features from existing ones (runs only if the needed columns exist).

In [9]:
df_fe = df_fs.copy()

if 'Age' in df_fe.columns:
    df_fe['AgeGroup'] = pd.cut(df_fe['Age'], bins=[0, 12, 18, 35, 60, 120], labels=['Child','Teen','Young','Adult','Senior'])

if 'SibSp' in df_fe.columns and 'Parch' in df_fe.columns:
    df_fe['FamilySize'] = df_fe['SibSp'] + df_fe['Parch'] + 1

if 'Fare' in df_fe.columns:
    df_fe['LogFare'] = np.log1p(df_fe['Fare'])

df_fe.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Age__was_missing,Cabin__was_missing,Embarked__was_missing,AgeGroup,FamilySize,LogFare
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1.0,0,A/5 21171,7.2500,B96 B98,S,0,1,0,Young,2.0,2.110213
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1.0,0,PC 17599,65.6344,C85,C,0,0,0,Adult,2.0,4.199221
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0.0,0,STON/O2. 3101282,7.9250,B96 B98,S,0,1,0,Young,1.0,2.188856
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1.0,0,113803,53.1000,C123,S,0,0,0,Young,2.0,3.990834
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0.0,0,373450,8.0500,B96 B98,S,0,1,0,Young,1.0,2.202765


## 1.4 One-Hot Encoding / Dummy Coding

Convert **nominal** categorical columns into 0/1 columns using `pd.get_dummies`.

In [10]:
df_ohe = df_fe.copy()
cat_cols = df_ohe.select_dtypes(include=['object', 'string', 'category']).columns.tolist()

# For a simple demo, encode only a few categorical columns
cat_cols = cat_cols[:5]
df_ohe = pd.get_dummies(df_ohe, columns=cat_cols, drop_first=True)
df_ohe.shape

(891, 1732)

## 1.5 Ordinal Encoding / Label Encoding

- **Ordinal encoding**: use when categories have a natural order (Low < Medium < High).
- **Label encoding**: often used for a categorical target.

In [11]:
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder

demo = pd.DataFrame({'Quality': ['Low', 'Medium', 'High', 'Low', 'High']})
ord_enc = OrdinalEncoder(categories=[['Low', 'Medium', 'High']])
demo['Quality_encoded'] = ord_enc.fit_transform(demo[['Quality']]).astype(int)
demo

,Quality,Quality_encoded
0,Low,0
1,Medium,1
2,High,2
3,Low,0
4,High,2


In [12]:
df_lbl = df_fe.copy()
target_candidates = ['Survived', 'target', 'Target', 'label', 'y']
target = next((c for c in target_candidates if c in df_lbl.columns), None)

if target is not None and not np.issubdtype(df_lbl[target].dtype, np.number):
    le = LabelEncoder()
    df_lbl[target] = le.fit_transform(df_lbl[target].astype(str))
    print('Encoded target:', target)
    print('Classes:', list(le.classes_))
else:
    print('No categorical target to label-encode (this is OK).')

No categorical target to label-encode (this is OK).


## 1.6 Data Transformation

Common transformations:
- **Standardization** (StandardScaler)
- **Normalization** (MinMaxScaler)
- **Log transform** (for skewed variables, e.g., `log1p`)

In [13]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

df_tr = df_ohe.copy()
num_cols = df_tr.select_dtypes(include=[np.number]).columns.tolist()

# Do not scale the target (if present)
target_candidates = ['Survived', 'target', 'Target', 'label', 'y']
target = next((c for c in target_candidates if c in df_tr.columns), None)
scale_cols = [c for c in num_cols if c != target]

df_std = df_tr.copy()
if len(scale_cols) > 0:
    scaler = StandardScaler()
    df_std[scale_cols] = scaler.fit_transform(df_std[scale_cols])
df_std[scale_cols].describe().T[['mean','std']].head(10) if len(scale_cols) > 0 else 'No numeric columns to scale.'

,mean,std
PassengerId,6.379733e-17,1.000562
Pclass,-8.772133e-17,1.000562
Age,2.392400e-17,1.000562
SibSp,1.196200e-17,1.000562
Parch,5.382900e-17,1.000562
Fare,9.968332e-17,1.000562
Age__was_missing,-1.644775e-17,1.000562
Cabin__was_missing,-9.569599e-17,1.000562
Embarked__was_missing,3.189866e-17,1.000562
FamilySize,-7.974666e-18,1.000562


In [14]:
df_mm = df_tr.copy()
if len(scale_cols) > 0:
    mm = MinMaxScaler()
    df_mm[scale_cols] = mm.fit_transform(df_mm[scale_cols])
df_mm[scale_cols].describe().T[['min','max']].head(10) if len(scale_cols) > 0 else 'No numeric columns to scale.'

,min,max
PassengerId,0.0,1.0
Pclass,0.0,1.0
Age,0.0,1.0
SibSp,0.0,1.0
Parch,0.0,1.0
Fare,0.0,1.0
Age__was_missing,0.0,1.0
Cabin__was_missing,0.0,1.0
Embarked__was_missing,0.0,1.0
FamilySize,0.0,1.0


## Save Preprocessed Dataset

We export one preprocessed version (selection + derived features + one-hot encoding).

In [15]:
out_path = 'output/1.10_preprocessed_train.csv'
df_ohe.to_csv(out_path, index=False)
out_path

'output/1.10_preprocessed_train.csv'